# Analysing Pilot Annotation Results

## Summary of findings:

1. Pilot reliability (do the annotators agree with each other?)
- α = 0.68 implies they have moderate agreement, but the estimate is not informative in a small sample (10 items). 
- 95% confidence interval is [0.04,  0.96] and P(α>0.80) = 0.26. So α = 0.68 cannot be used as a reliability metric. 
- The pilot run was used as a training round, and we will only report the α for the full 50-item run.

2. Pilot validity (do the annotators match the intended scores?)
- They match my intended scores better than they match each other (validity > agreement). 
- Weighted agreement with gold score: Rubayet = 1.000, Jesi = 0.968, Yousuf = 0.956
- Bias is small and not systematic. Nobody is consistently lenient or strict as there is no consistent gap in any direction. 
- We can conclude that my written guidelines are good enough. Any disagreements are about specific items, not because any person is interpreting the rubric differently.

3. Inspect disagreements
- The widest disagreements (spread >1) are due to 1 person (Yousuf) not catching the factual errors on tasks DIAG-07 and PRES-03. T
Action: make the guidelines more explicit and nudge Yousuf
- For tasks where the caveats are missing (DIAG-06 and PRES-01), 2 annotators were 1 point off from the intended score. This is okay and I accept their rationales. Agreement =/= correctness. 
Action: Standardised rule: deduct 1 point for each missing caveat or wrong figure.
- 5 out of 10 items unanimously matched the gold scores, including both extremes (0 and 5). This suggests that the definitions of “completely wrong” and “completely right” are clear. The disagreements are for the subjective cases (middle of the scale) as expected, which justifies the need for an LLM-judge.

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
df_anno = pd.read_csv('pilot_annotator_scores.csv')
df_anno.head()

,tagger_name,task_id,score,comments,flag_comment,seconds_on_task,seconds_session_elapsed,submitted_at,session_id,presentation_order,received_at
0,Jesi,DIAG-05,5,The Agent's answer is complete. It has all the...,NaN,185,1945,2026-07-29T08:21:19.913Z,ms5s9uisg9kl6,1,7/29/2026 9:21:23
1,Jesi,PRES-01,4,"It has the key values (stats, p-value, rate) r...",NaN,391,2335,2026-07-29T08:27:50.432Z,ms5s9uisg9kl6,2,7/29/2026 9:27:53
2,Jesi,PRES-05,2,A lot of key information missing here (p-value...,NaN,368,2704,2026-07-29T08:33:58.775Z,ms5s9uisg9kl6,3,7/29/2026 9:34:01
3,Jesi,DIAG-06,4,Agent does answer the question accurately but ...,NaN,288,2992,2026-07-29T08:38:47.224Z,ms5s9uisg9kl6,4,7/29/2026 9:38:49
4,Jesi,PRES-02,0,The Agent does not mention anything close to t...,The first line in the answer was a bit confusi...,313,3305,2026-07-29T08:44:00.596Z,ms5s9uisg9kl6,5,7/29/2026 9:44:03


In [6]:
df_intend = pd.read_csv('pilot_intended_scores.csv')
df_intend.head()

,task_id,question,gold_answer,agent_answer,intended_score,example comments
0,DIAG-05,What are the top 2 drivers of dissatisfaction ...,TradeA's two biggest drivers of dissatisfactio...,"Two things stand out for TradeA, and they're e...",5,"Correct — verdict, both figures, and the ""too ..."
1,DIAG-06,What are TradeA's top 2 priority issues based ...,TradeA's top two priority issues are the app o...,TradeA's priority list comes out as: 1) app or...,3,Catch omitted: presents a ranked order without...
2,DIAG-07,What are the top 2 drivers of dissatisfaction ...,Competitor is ConsultA's clear top driver of d...,Competitor is the clear problem for ConsultA a...,4,Minor figure error: sample size given as 12 ra...
3,DIAG-08,What are ConsultA's top 2 priority issues base...,Competitor is ConsultA's only genuine priority...,ConsultA has a couple of areas showing up in t...,1,"Vague: names no issue, gives no priority score..."
4,PRES-01,How do TradeA's complaint drivers compare agai...,The app or website is TradeA's real weakness; ...,"Against the Trading benchmark, TradeA splits i...",5,"Correct — both directions right, both p-values..."


---
## 1. Reshape to a units x raters reliability matrix

In [7]:
SCALE = list(range(0, 6))          # ordered categories 0..5
K = len(SCALE)                      # K = 6 levels

for _df in (df_anno, df_intend):    # strip BOM / stray whitespace from headers
    _df.columns = _df.columns.str.replace('﻿', '', regex=False).str.strip()

# units (task_id) x raters (tagger_name)
wide = df_anno.pivot(index='task_id', columns='tagger_name', values='score')
gold = df_intend.set_index('task_id')['intended_score'].reindex(wide.index)

RATERS = list(wide.columns)
M = wide.to_numpy(dtype=float)      # reliability matrix
n_units, R = M.shape

print(f'{n_units} units x {R} raters, scale 0-{K-1}, missing cells: {int(np.isnan(M).sum())}')
wide.assign(intended=gold)

10 units x 3 raters, scale 0-5, missing cells: 0


tagger_name,Jesi,Rubayet Bushra,Yousuf,intended
task_id,,,,
DIAG-05,5,5,5,5
DIAG-06,4,3,4,3
DIAG-07,2,4,5,4
DIAG-08,1,1,1,1
PRES-01,4,5,4,5
PRES-02,0,0,0,0
PRES-03,4,3,5,3
PRES-04,4,4,4,4
PRES-05,2,2,2,2


---
## 2. Pilot reliability: ordinal Krippendorff's alpha

Inter-annotator agreement, chance-corrected, on an ordered scale.
`alpha = 1 - D_o / D_e` computed from the coincidence matrix, with Krippendorff's
ordinal difference function `d2[c,k] = (sum(n_g for g in c..k) - (n_c + n_k)/2)**2`.

Because n = 10 units is small, two extras are reported:
- `alpha_U`, the unbiased-`I_e` variant (Martin Andres & Alvarez Hernandez 2024, eqs. 33 & 36).
  The paper's own guidance is that the correction is worth applying while n <= 33.
- a percentile bootstrap CI over units, since point estimates swing hard at this n.

In [8]:
def krippendorff_alpha_ordinal(matrix, scale=SCALE):
    """Krippendorff's alpha with the ordinal difference function. matrix = units x raters."""
    pos = {c: i for i, c in enumerate(scale)}
    k = len(scale)

    # coincidence matrix: each unit contributes its m*(m-1) ordered pairs, weighted 1/(m-1)
    o = np.zeros((k, k))
    for row in matrix:
        vals = [v for v in row if not np.isnan(v)]
        m = len(vals)
        if m < 2:                      # unpairable unit, dropped
            continue
        for a in vals:
            for b in vals:
                o[pos[a], pos[b]] += 1.0 / (m - 1)
        for a in vals:                 # remove the self-pairs added above
            o[pos[a], pos[a]] -= 1.0 / (m - 1)

    n_c = o.sum(axis=1)
    n = n_c.sum()
    if n < 2:
        return np.nan

    # ordinal metric: distance depends on the mass lying between the two levels
    d2 = np.zeros((k, k))
    for c in range(k):
        for kk in range(k):
            lo, hi = min(c, kk), max(c, kk)
            d2[c, kk] = (n_c[lo:hi + 1].sum() - (n_c[c] + n_c[kk]) / 2) ** 2

    D_o = (o * d2).sum() / n
    D_e = (np.outer(n_c, n_c) * d2).sum() / (n * (n - 1))
    return 1 - D_o / D_e


def small_sample_correct(alpha, n, R):
    """Unbiased-I_e version of alpha. Martin Andres & Alvarez Hernandez (2024) eqs. 33 & 36."""
    k_F = (2 * n * alpha - 1) / (2 * n - 1)                                    # strip Krippendorff's I_o correction
    k_FU = ((R * n - 1) * k_F + 1) / ((R - 1) * k_F + (R * (n - 1) + 1))       # eq. 33
    return ((2 * n - 1) * k_FU + 1) / (2 * n)                                  # eq. 36


alpha = krippendorff_alpha_ordinal(M)
alpha_U = small_sample_correct(alpha, n_units, R)

rng = np.random.default_rng(0)
boot = np.array([krippendorff_alpha_ordinal(M[rng.integers(0, n_units, n_units)])
                 for _ in range(5000)])
lo, hi = np.nanpercentile(boot, [2.5, 97.5])

print(f'ordinal Krippendorff alpha   = {alpha:.3f}')
print(f'  small-sample corrected     = {alpha_U:.3f}   (n={n_units}, R={R})')
print(f'  95% bootstrap CI           = [{lo:.3f}, {hi:.3f}]   over {n_units} units')
print(f'  P(alpha >= 0.80) = {np.mean(boot >= 0.80):.2f}   P(alpha >= 0.67) = {np.mean(boot >= 0.67):.2f}')

ordinal Krippendorff alpha   = 0.677
  small-sample corrected     = 0.702   (n=10, R=3)
  95% bootstrap CI           = [0.035, 0.957]   over 10 units
  P(alpha >= 0.80) = 0.26   P(alpha >= 0.67) = 0.52


---
## 3. Pilot gold check: weighted pairwise agreement with the intended score

Agreement is not validity (Baledent et al. 2022, 4.1) — annotators can converge on a
shared misreading of the guidelines. So each annotator is also scored against my intended
scores, pairwise, using Fleiss-Cohen quadratic weights `w_ij = 1 - ((i-j)/(K-1))**2`.

Reported per annotator:
- `w_agree` — quadratic-weighted agreement with intended (1.0 = identical)
- `exact` — proportion of exact matches
- `imperfection` — RMS distance from intended (Baledent formula 1; lower is better)
- `bias` — mean signed error, i.e. lenient (+) vs strict (-) against my intended scores

In [9]:
def quad_w(i, j, k=K):
    """Fleiss-Cohen quadratic weights on an ordered scale."""
    return 1 - ((i - j) / (k - 1)) ** 2


g = gold.to_numpy(dtype=float)

gold_check = pd.DataFrame({
    'w_agree':      [np.nanmean(quad_w(wide[r].to_numpy(dtype=float), g)) for r in RATERS],
    'exact':        [np.nanmean(wide[r].to_numpy(dtype=float) == g) for r in RATERS],
    'imperfection': [np.sqrt(np.nanmean((wide[r].to_numpy(dtype=float) - g) ** 2)) for r in RATERS],
    'bias':         [np.nanmean(wide[r].to_numpy(dtype=float) - g) for r in RATERS],
}, index=RATERS).round(3)

# the panel's own consensus vs intended, as a group-level validity check
consensus = np.nanmedian(M, axis=1)
gold_check.loc['— panel median —'] = [
    np.mean(quad_w(consensus, g)), np.mean(consensus == g),
    np.sqrt(np.mean((consensus - g) ** 2)), np.mean(consensus - g),
]

print(f'group imperfection = {gold_check.loc[RATERS, "imperfection"].mean():.3f}'
      f'   (Baledent formula 2)')
gold_check.round(3)

group imperfection = 0.648   (Baledent formula 2)


,w_agree,exact,imperfection,bias
Jesi,0.968,0.5,0.894,-0.2
Rubayet Bushra,1.000,1.0,0.000,0.0
Yousuf,0.956,0.5,1.049,0.1
— panel median —,0.984,0.6,0.632,0.0


In [10]:
# Consensuality (Baledent formulas 3 & 4): does removing this annotator raise group
# disagreement? Positive = they pull the panel together. Pairs with the columns above:
# high consensuality + high imperfection = confidently wrong, the case worth catching.
def disagreement(cols):
    return np.nanmean(np.nanstd(wide[cols].to_numpy(dtype=float), axis=1))


d_all = disagreement(RATERS)
gold_check.loc[RATERS, 'consensuality'] = [
    disagreement([c for c in RATERS if c != r]) - d_all for r in RATERS
]

print(f'group disagreement = {d_all:.3f}')
gold_check.round(3)

group disagreement = 0.382


,w_agree,exact,imperfection,bias,consensuality
Jesi,0.968,0.5,0.894,-0.2,-0.032
Rubayet Bushra,1.000,1.0,0.000,0.0,-0.132
Yousuf,0.956,0.5,1.049,0.1,-0.082
— panel median —,0.984,0.6,0.632,0.0,NaN


---
## 4. Disagreement inspection

Two failure modes, flagged separately, because they call for different fixes:
- `split` — annotators disagree with each other (spread >= 2). A guideline ambiguity.
- `off_gold` — the panel converges somewhere other than my intended score
  (|panel median - intended| >= 1). Either my intended score is wrong, or the
  guideline says something I did not mean it to say.

In [11]:
items = wide.copy()
items['intended'] = gold
items['median'] = consensus
items['spread'] = np.nanmax(M, axis=1) - np.nanmin(M, axis=1)
items['max_dev'] = np.nanmax(np.abs(M - g[:, None]), axis=1)
items['gold_gap'] = items['median'] - items['intended']
items['split'] = items['spread'] >= 2
items['off_gold'] = items['gold_gap'].abs() >= 1

items.sort_values(['spread', 'max_dev'], ascending=False)

tagger_name,Jesi,Rubayet Bushra,Yousuf,intended,median,spread,max_dev,gold_gap,split,off_gold
task_id,,,,,,,,,,
DIAG-07,2,4,5,4,4.0,3.0,2.0,0.0,True,False
PRES-03,4,3,5,3,4.0,2.0,2.0,1.0,True,True
PRES-06,4,5,3,5,4.0,2.0,2.0,-1.0,True,True
DIAG-06,4,3,4,3,4.0,1.0,1.0,1.0,False,True
PRES-01,4,5,4,5,4.0,1.0,1.0,-1.0,False,True
DIAG-05,5,5,5,5,5.0,0.0,0.0,0.0,False,False
DIAG-08,1,1,1,1,1.0,0.0,0.0,0.0,False,False
PRES-02,0,0,0,0,0.0,0.0,0.0,0.0,False,False
PRES-04,4,4,4,4,4.0,0.0,0.0,0.0,False,False


In [12]:
flagged = items[items['split'] | items['off_gold']]
print(f'{len(flagged)} of {n_units} items flagged for review\n')

meta = df_intend.set_index('task_id')
notes = df_anno.set_index(['task_id', 'tagger_name'])

for tid, row in flagged.iterrows():
    tags = ' + '.join([t for t, on in [('split', row['split']), ('off_gold', row['off_gold'])] if on])
    print('=' * 100)
    print(f'{tid}  [{tags}]   intended={int(row["intended"])}  median={row["median"]:.1f}  spread={int(row["spread"])}')
    print(f'Q: {meta.loc[tid, "question"]}')
    print(f'\n  intended rationale: {meta.loc[tid, "example comments"]}')
    for r in RATERS:
        print(f'\n  [{int(row[r])}] {r}: {notes.loc[(tid, r), "comments"]}')
    print()

5 of 10 items flagged for review

DIAG-06  [off_gold]   intended=3  median=4.0  spread=1
Q: What are TradeA's top 2 priority issues based on dissatisfaction impact?

  intended rationale: Catch omitted: presents a ranked order without noting the two are tied and either order is acceptable. Figures right; the caveat that makes the ranking meaningful is missing.

  [4] Jesi: Agent does answer the question accurately but does not go into detail on what could be the caveat behind the numbers

  [3] Rubayet Bushra: Relevant and correct direction. The agent correctly mentions top two priority issues and scores but it misses the catch that two are tied on priority score, so either order is acceptable; neither can be ranked above the other.

  [4] Yousuf: Relevant, Accurate and Complete but missing ranking

DIAG-07  [split]   intended=4  median=4.0  spread=3
Q: What are the top 2 drivers of dissatisfaction for ConsultA?

  intended rationale: Minor figure error: sample size given as 12 rather 

In [18]:
# --- Timing ---
t = df_anno.copy()
t['submitted_at'] = pd.to_datetime(t['submitted_at'], utc=True)

timing = t.groupby('tagger_name').agg(
    items=('task_id', 'count'),
    on_task_min=('seconds_on_task', lambda s: s.sum() / 60),      # time actually spent grading
    median_item_s=('seconds_on_task', 'median'),
    slowest_item_s=('seconds_on_task', 'max'),
    session_min=('seconds_session_elapsed', lambda s: s.max() / 60),  # wall-clock, incl. breaks
    first_submit=('submitted_at', 'min'),
    last_submit=('submitted_at', 'max'),
)
timing['idle_min'] = timing['session_min'] - timing['on_task_min']
for c in ('first_submit', 'last_submit'):
    timing[c] = timing[c].dt.strftime('%d %b %H:%M')

start, end = t['submitted_at'].min(), t['submitted_at'].max()
print(f'total on-task time, all taggers : {t["seconds_on_task"].sum() / 60:.1f} min')
print(f'pilot span                      : {start:%Y-%m-%d %H:%M} -> {end:%Y-%m-%d %H:%M} UTC '
      f'({(end - start).total_seconds() / 3600:.1f} h)')
timing.round(1)

total on-task time, all taggers : 276.5 min
pilot span                      : 2026-07-28 18:04 -> 2026-07-29 09:45 UTC (15.7 h)


,items,on_task_min,median_item_s,slowest_item_s,session_min,first_submit,last_submit,idle_min
tagger_name,,,,,,,,
Jesi,10,47.9,281.5,397,77.2,29 Jul 08:21,29 Jul 09:06,29.4
Rubayet Bushra,10,124.9,638.0,1495,124.9,28 Jul 18:04,28 Jul 19:46,-0.0
Yousuf,10,103.8,342.0,2833,113.0,29 Jul 08:04,29 Jul 09:45,9.3
